# 02: Transfer Learning, Feature Extraction & Performance Logging

**Track 09: Computer Vision & Convolutional Neural Networks** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Fine-tune deep vision backbones (MobileNet/ResNet), freeze feature extractors, customize classification heads, and monitor metrics.


## 1. Transfer Learning Strategy
We freeze base convolutional weights $\nabla W_{\text{backbone}} = 0$ and train only task-specific linear heads.

In [ ]:
import torch
import torch.nn as nn

# Base pretrained-style feature extractor backbone
class PretrainedBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(64, 1000)

    def forward(self, x):
        feat = self.features(x)
        flattened = feat.view(feat.size(0), -1)
        return self.classifier(flattened)

backbone = PretrainedBackbone()

# Freeze feature extractor layers (transfer learning)
for param in backbone.features.parameters():
    param.requires_grad = False

# Replace classifier head for 3 custom target classes
in_features = 64
backbone.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, 3) # 3 custom output classes
)

trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in backbone.parameters())

print("=== Transfer Learning Architecture Setup ===")
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")

## 2. Forward Pass Verification
Verify output logits shape on an input RGB batch $(N, 3, 224, 224)$.

In [ ]:
dummy_images = torch.randn(4, 3, 224, 224)
output_logits = backbone(dummy_images)
probabilities = torch.softmax(output_logits, dim=1)

print(f"Input Shape : {dummy_images.shape}")
print(f"Output Logits Shape : {output_logits.shape}")
print("Predicted Class Probabilities:")
print(probabilities.detach().numpy().round(4))